# Treasury Auctions -- monthly update

Re-fetches a buffer window before the last known issue_date, with no
upper bound -- the archive already includes announced-but-not-yet-issued
future auctions, and capping the refetch at "today" would silently drop
those (confirmed: an earlier version of this notebook lost 6 real rows
this way). Replaces that window in the clean dataset and re-saves.

In [1]:
import os
import pandas as pd
from datetime import date, timedelta
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "Helpers"))
from treasury_fiscaldata_helpers import fetch_auctions

BASE = os.path.dirname(os.getcwd())
RAW_PATH = os.path.join(BASE, "Data Input", "Raw Input", "treasury_auctions_raw.csv")
CLEAN_PATH = os.path.join(BASE, "Data Output", "treasury_auctions_clean.csv")
BUFFER_DAYS = 30

## Where are we

In [2]:
existing = pd.read_csv(CLEAN_PATH, parse_dates=["auction_date", "issue_date", "maturity_date", "announcemt_date"])
last_date = existing["issue_date"].max()
print(f"Last issue_date on file: {last_date.date()}")

Last issue_date on file: 2026-10-15


## Re-fetch the buffer window, no upper bound

In [3]:
refetch_start = (last_date - timedelta(days=BUFFER_DAYS)).date().isoformat()
print(f"Re-fetching {refetch_start} -> latest available")

fresh = fetch_auctions(start=refetch_start, end=None, include_extra=True)
print(f"{len(fresh):,} rows in the refetched window")

Re-fetching 2026-09-15 -> latest available


40 rows in the refetched window


## Replace the window and save

In [4]:
existing_raw = pd.read_csv(RAW_PATH, parse_dates=["auction_date", "issue_date", "maturity_date", "announcemt_date"])
kept_raw = existing_raw[existing_raw["issue_date"] < pd.Timestamp(refetch_start)]
updated_raw = pd.concat([kept_raw, fresh], ignore_index=True).drop_duplicates().sort_values("issue_date").reset_index(drop=True)
updated_raw.to_csv(RAW_PATH, index=False)

updated_clean = updated_raw.dropna(subset=["cusip"]).drop_duplicates(subset=["cusip", "issue_date"]).sort_values("issue_date").reset_index(drop=True)
updated_clean.to_csv(CLEAN_PATH, index=False)

print(f"{len(updated_raw):,} raw rows ({len(updated_raw) - len(existing_raw):,} net new)")
print(f"{len(updated_clean):,} clean rows ({len(updated_clean) - len(existing):,} net new)")
print(f"Saved to: {RAW_PATH}")
print(f"Saved to: {CLEAN_PATH}")

11,143 raw rows (0 net new)
11,143 clean rows (0 net new)
Saved to: C:\Users\User\Desktop\DISSERTATION PAPERS\FED_TREASURY_DATA\Data Input\Raw Input\treasury_auctions_raw.csv
Saved to: C:\Users\User\Desktop\DISSERTATION PAPERS\FED_TREASURY_DATA\Data Output\treasury_auctions_clean.csv
